# Notebook 1 — Data exploration and splitting

**Computer Vision CW1 — Diabetic Retinopathy Stage Detection**
Mohamed Shafran · NIBM (BSCCOMP24.2P)

This notebook is the first step of the project. It does four things:

1. **Finds the APTOS 2019 data** and loads the labels.
2. **Measures every image** — size, brightness, contrast and sharpness — and checks whether image quality or camera differs between stages, so the dataset section of the report is based on numbers, not guesses.
3. **Finds duplicate and near-duplicate images**, which the dataset is known to contain.
4. **Writes one fixed train / validation / test split** to `splits.csv`. Every later notebook reads that file, so all experiments are compared on exactly the same data.

**How to run it:** set *Accelerator* to **None** (no GPU needed here — it saves your weekly quota), attach the APTOS 2019 competition data, then **Run All**. It takes a few minutes, most of it reading the 3,662 images once.

**No model is trained here, and the test split is never looked at again until the final evaluation.**

In [ ]:
# ============================================================
# Setup: imports, fixed seed, output folders, one shared style
# ============================================================
import os, io, json, time, hashlib
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import cv2
from PIL import Image

SEED = 42                      # fixed, so the split is identical every time this runs
np.random.seed(SEED)

OUT_DIR = Path("/kaggle/working")          # Kaggle keeps what we save here
FIG_DIR = OUT_DIR / "figures"
FIG_DIR.mkdir(parents=True, exist_ok=True)

CLASS_NAMES = ["No DR", "Mild", "Moderate", "Severe", "Proliferative"]

# One colour scheme for every figure in the project.
# The five stages are ORDERED (0 -> 4 = healthy -> worst), so they get a single
# hue from light to dark rather than five unrelated colours. Splits are not
# ordered, so they get three distinct hues.
STAGE_COLORS = ["#86b6ef", "#5598e7", "#2a78d6", "#1c5cab", "#104281"]
SPLIT_COLORS = {"train": "#2a78d6", "val": "#eb6834", "test": "#1baf7a"}
INK, INK_SOFT, MUTED, GRID, FLAG = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#d03b3b"

plt.rcParams.update({
    "figure.facecolor": "#fcfcfb", "axes.facecolor": "#fcfcfb",
    "axes.edgecolor": "#c3c2b7", "axes.labelcolor": INK_SOFT,
    "xtick.color": MUTED, "ytick.color": MUTED,
    "axes.titlesize": 12, "axes.titleweight": "bold",
    "font.size": 10, "figure.dpi": 110, "savefig.bbox": "tight",
})

def tidy(ax):
    """Remove chart junk: drop the top/right frame, put a hairline grid behind the data."""
    for side in ("top", "right"):
        ax.spines[side].set_visible(False)
    ax.set_axisbelow(True)
    ax.grid(axis="y", color=GRID, linewidth=0.8)
    return ax

print("OpenCV", cv2.__version__, "| NumPy", np.__version__, "| pandas", pd.__version__)

## 1. Find the data and load the labels

Kaggle mounts competition data under a folder whose name we should not assume — on this account it sits at `/kaggle/input/competitions/aptos2019-blindness-detection/`. Instead of hard-coding that path, the function below searches for the folder that actually contains `train.csv`. If Kaggle changes the layout, the notebook still works.

`train.csv` has one row per image: `id_code` (the file name without `.png`) and `diagnosis` (the stage, 0–4).

In [ ]:
def find_data_dir(root="/kaggle/input"):
    """Return the folder that holds train.csv, wherever Kaggle mounted it."""
    for path, dirs, files in os.walk(root):
        if "train.csv" in files:
            return Path(path)
        # never crawl into the image folders while searching - thousands of files
        dirs[:] = [d for d in dirs if not d.lower().endswith("_images")]
    raise FileNotFoundError("train.csv not found under /kaggle/input")

DATA = find_data_dir()
TRAIN_IMG_DIR = DATA / "train_images"
print("Data folder:", DATA)

labels = pd.read_csv(DATA / "train.csv")
labels["path"] = labels["id_code"].map(lambda code: str(TRAIN_IMG_DIR / f"{code}.png"))

missing = [p for p in labels["path"] if not os.path.exists(p)]
print("rows in train.csv :", len(labels))
print("image files found :", len(labels) - len(missing))
assert not missing, f"{len(missing)} images listed in train.csv are missing from disk"

labels.head()

## 2. Class distribution

The first thing to establish is how unbalanced the dataset is, because it shapes almost every later decision: which metric we judge the model by, whether we weight the classes, and how much confidence the rare-stage results deserve.

In [ ]:
counts = labels["diagnosis"].value_counts().sort_index()
dist = pd.DataFrame({
    "stage":   counts.index,
    "name":    [CLASS_NAMES[i] for i in counts.index],
    "images":  counts.values,
    "percent": (100 * counts.values / len(labels)).round(1),
})
print(dist.to_string(index=False))
print("\ntotal images       :", len(labels))
print(f"largest / smallest : {counts.max()/counts.min():.1f}x "
      f"({CLASS_NAMES[int(counts.idxmax())]} vs {CLASS_NAMES[int(counts.idxmin())]})")

In [ ]:
# Figure 1 - how many images each stage has
fig, ax = plt.subplots(figsize=(7.2, 4.2))
bars = ax.bar(dist["stage"], dist["images"], width=0.66, color=STAGE_COLORS)
for bar, n, pct in zip(bars, dist["images"], dist["percent"]):
    ax.text(bar.get_x() + bar.get_width()/2, n + counts.max()*0.02,
            f"{n}\n{pct}%", ha="center", va="bottom", fontsize=9, color=INK_SOFT)

ax.set_xticks(dist["stage"])
ax.set_xticklabels([f"{s}\n{n}" for s, n in zip(dist["stage"], dist["name"])])
ax.set_ylabel("Images")
ax.set_ylim(0, counts.max() * 1.20)
ax.set_title("APTOS 2019 - images per diabetic retinopathy stage")
tidy(ax)
plt.savefig(FIG_DIR / "fig01_class_distribution.png")
plt.show()

## 3. Measure every image

Each image is read **once**. Before anything is measured, the black border is cut away so
only the circular retina remains — otherwise the background would drag every brightness and
sharpness number towards zero, and by different amounts for different cameras.

| Measurement | What it tells us |
|---|---|
| width, height | how much the cameras differ, and how much resizing we will do |
| crop_fraction | how much of the frame is actually retina |
| brightness | mean grey value of the retina — low means underexposed |
| contrast | spread of grey values — low means flat and washed out |
| sharpness | variance of the Laplacian, a standard blur measure: sharp edges give a high value |
| md5 | the file's fingerprint — identical bytes mean the identical file |
| thumbnail | a 128×128 fingerprint of the *content*, used to find near-duplicates in section 4 |

Two speed details: `cv2.IMREAD_REDUCED_COLOR_4` decodes at quarter size, which is far faster and
plenty for statistics, and the true width and height still come from the file header. Every image
is then resized to the same 256×256 before measuring, so the numbers compare fairly across cameras.

**The content thumbnail — additional knowledge.** The 128×128 grey crop is stored *minus a blurred
copy of itself*. Every fundus photo shares the same broad shape — bright in the middle, dark at the
corners, a bright optic disc — and that shared shape would otherwise dominate any comparison.
Subtracting the blur removes it and leaves each eye's own vessels and lesions. Section 4 explains
why this matters.

In [ ]:
def crop_to_retina(bgr, tol=10):
    """Cut away the black border so only the circular retina is left.
    'tol' is the grey level below which a pixel counts as background."""
    gray = cv2.cvtColor(bgr, cv2.COLOR_BGR2GRAY)
    mask = gray > tol
    if not mask.any():
        return bgr                              # completely black - leave it alone
    rows = np.where(mask.any(axis=1))[0]
    cols = np.where(mask.any(axis=0))[0]
    return bgr[rows[0]:rows[-1] + 1, cols[0]:cols[-1] + 1]


def content_thumbnail(gray256):
    """128x128 fingerprint of what is INSIDE this eye.
    Subtracting a blurred copy removes the bright-disc-on-black shape that every
    fundus photo shares, leaving the vessels and lesions that differ."""
    thumb = cv2.resize(gray256, (128, 128), interpolation=cv2.INTER_AREA).astype(np.float32)
    return thumb - cv2.GaussianBlur(thumb, (0, 0), 1)


def dhash(gray256):
    """64-bit difference hash - kept only to show, in section 4, why it fails here."""
    small = cv2.resize(gray256, (9, 8), interpolation=cv2.INTER_AREA)
    return np.packbits(small[:, 1:] > small[:, :-1])


def scan_image(path):
    """Read one image once and return its measurements and its content thumbnail."""
    raw = open(path, "rb").read()

    with Image.open(io.BytesIO(raw)) as im:               # header only, no full decode
        width, height = im.size

    small = cv2.imdecode(np.frombuffer(raw, np.uint8), cv2.IMREAD_REDUCED_COLOR_4)
    if small is None:
        raise ValueError(f"could not decode {path}")

    retina = crop_to_retina(small)                        # black border removed first
    standard = cv2.resize(retina, (256, 256), interpolation=cv2.INTER_AREA)
    gray = cv2.cvtColor(standard, cv2.COLOR_BGR2GRAY)

    measurements = {
        "width": width, "height": height,
        "megapixels": round(width * height / 1e6, 2),
        "file_mb": round(len(raw) / 1e6, 3),
        "crop_fraction": round(retina.size / small.size, 3),          # 1.0 = no black border
        "brightness": float(gray.mean()),                             # 0 = black, 255 = white
        "contrast": float(gray.std()),
        "sharpness": float(cv2.Laplacian(gray, cv2.CV_64F).var()),    # low = blurred
        "md5": hashlib.md5(raw).hexdigest(),                          # identical FILES
        "dhash": dhash(gray).tobytes().hex(),
    }
    return measurements, content_thumbnail(gray)

In [ ]:
# Read all images. Threads help because OpenCV releases the interpreter lock while
# decoding, so several images are decoded at the same time.
start = time.time()
with ThreadPoolExecutor(max_workers=8) as pool:
    results = list(pool.map(scan_image, labels["path"]))

records = [r[0] for r in results]
thumbs  = np.stack([r[1] for r in results])          # (N, 128, 128) content fingerprints
stats   = pd.concat([labels.reset_index(drop=True), pd.DataFrame(records)], axis=1)

print(f"scanned {len(stats)} images in {time.time() - start:.0f} s")
print("thumbnail array:", thumbs.shape)

stats[["width", "height", "megapixels", "file_mb", "crop_fraction",
       "brightness", "contrast", "sharpness"]].describe().round(2)

In [ ]:
# Figure 2 - what sizes do the images come in?
sizes = (stats.groupby(["width", "height"]).size()
         .reset_index(name="count").sort_values("count", ascending=False))
print("distinct image sizes:", len(sizes))
print(sizes.head(5).to_string(index=False))

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(sizes["width"], sizes["height"],
           s=20 + 240 * sizes["count"] / sizes["count"].max(),
           color="#2a78d6", alpha=0.55, edgecolor="#fcfcfb", linewidth=0.8)
for _, row in sizes.head(3).iterrows():
    ax.annotate(f'{int(row.width)}x{int(row.height)}  ({int(row["count"])})',
                (row.width, row.height), textcoords="offset points",
                xytext=(9, 6), fontsize=9, color=INK_SOFT)

ax.set_xlabel("Width (pixels)")
ax.set_ylabel("Height (pixels)")
ax.set_title("Image sizes in the dataset (bubble area = how many images)")
tidy(ax)
ax.grid(axis="x", color=GRID, linewidth=0.8)
plt.savefig(FIG_DIR / "fig02_image_sizes.png")
plt.show()

In [ ]:
# Figure 3 - five random images from each stage
fig, axes = plt.subplots(5, 5, figsize=(12.5, 13))
for row in range(5):
    picks = stats[stats["diagnosis"] == row].sample(5, random_state=SEED)
    for col, (_, r) in enumerate(picks.iterrows()):
        image = cv2.cvtColor(cv2.imread(r["path"], cv2.IMREAD_REDUCED_COLOR_8), cv2.COLOR_BGR2RGB)
        ax = axes[row, col]
        ax.imshow(image)
        ax.set_xticks([]); ax.set_yticks([])
        for spine in ax.spines.values():
            spine.set_color(STAGE_COLORS[row]); spine.set_linewidth(2.5)
        if col == 0:
            ax.set_ylabel(f"{row} - {CLASS_NAMES[row]}", fontsize=10, color=INK)

fig.suptitle("Five random images per stage", fontsize=13, fontweight="bold", color=INK)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig03_samples_per_class.png")
plt.show()

### Image quality

Real screening photos vary: some are out of focus, some underexposed. The competition organisers say so themselves. Rather than repeat that claim, we measure it and keep the numbers for the report's dataset limitations section. Nothing is deleted here — the flags are used later, in the error analysis, to check whether the model's mistakes concentrate on poor images.

In [ ]:
# Flag the worst 5% by sharpness and by brightness
BLUR_CUT   = stats["sharpness"].quantile(0.05)
DARK_CUT   = stats["brightness"].quantile(0.05)
low_quality = (stats["sharpness"] <= BLUR_CUT) | (stats["brightness"] <= DARK_CUT)
stats["low_quality"] = low_quality          # kept in splits.csv for the error analysis

print(f"blurriest 5% cut-off : sharpness <= {BLUR_CUT:.0f}")
print(f"darkest 5% cut-off   : brightness <= {DARK_CUT:.0f}")
print(f"images flagged       : {int(low_quality.sum())} ({100*low_quality.mean():.1f}%)")

fig, ax = plt.subplots(figsize=(7.2, 4.6))
ax.scatter(stats.loc[~low_quality, "brightness"], stats.loc[~low_quality, "sharpness"],
           s=14, color="#2a78d6", alpha=0.35, linewidth=0, label="Normal")
ax.scatter(stats.loc[low_quality, "brightness"], stats.loc[low_quality, "sharpness"],
           s=16, color=FLAG, alpha=0.8, linewidth=0, label="Dark or blurred (worst 5%)")
ax.set_yscale("log")
ax.set_xlabel("Brightness (mean grey value)")
ax.set_ylabel("Sharpness (log scale)")
ax.set_title("Image quality - most images are usable, with a dark and blurred tail")
ax.legend(frameon=False, labelcolor=INK_SOFT)
tidy(ax)
plt.savefig(FIG_DIR / "fig04_image_quality.png")
plt.show()

In [ ]:
# Figure 5 - the six least sharp images, as evidence of what "low quality" means here
worst = stats.nsmallest(6, "sharpness")
fig, axes = plt.subplots(1, 6, figsize=(15, 3.2))
for ax, (_, r) in zip(axes, worst.iterrows()):
    ax.imshow(cv2.cvtColor(cv2.imread(r["path"], cv2.IMREAD_REDUCED_COLOR_8), cv2.COLOR_BGR2RGB))
    ax.set_title(f'{CLASS_NAMES[int(r["diagnosis"])]}\nsharpness {r["sharpness"]:.0f}',
                 fontsize=9, color=INK_SOFT)
    ax.axis("off")
fig.suptitle("The six least sharp images in the dataset", fontsize=12, fontweight="bold", color=INK)
plt.tight_layout()
plt.savefig(FIG_DIR / "fig05_worst_quality.png")
plt.show()

### Could the model take a shortcut?

A CNN learns whatever separates the classes most easily. If diseased eyes were photographed badly
more often, or mostly with one camera, the model could learn *"dark photo → disease"* or *"this
camera → this stage"* instead of the lesions themselves. This is called **shortcut learning**
(additional knowledge): the model looks accurate on this dataset and then fails on photographs from
anywhere else. The organisers state that the images *"were gathered from multiple clinics using a
variety of cameras over an extended period of time"*, so it is worth checking before any model is
trained.

The dataset has no camera column. The **image size** is the closest substitute, because images from
the same camera setup come out at the same size. Two questions, one figure:

1. Are dark or blurred photos more common in some stages than in others?
2. Does the mix of stages change from one image size to another?

Nothing is removed because of this check. It tells us what the preprocessing has to even out —
notebook 2 crops, resizes and normalises every image in the same way, which reduces the differences
between cameras — and what the error analysis must look at once a model exists.

In [ ]:
# --- 1. image quality per stage ---------------------------------------------
quality_by_stage = (stats.groupby("diagnosis")
                    .agg(images=("id_code", "size"),
                         median_brightness=("brightness", "median"),
                         median_sharpness=("sharpness", "median"),
                         low_quality_percent=("low_quality", "mean")))
quality_by_stage["low_quality_percent"] *= 100
print("image quality per stage:")
print(quality_by_stage.rename(index=lambda s: f"{s} {CLASS_NAMES[s]}").round(1).to_string())

# --- 2. stage mix per image size (our stand-in for the camera) --------------
stats["image_size"] = stats["width"].astype(str) + "x" + stats["height"].astype(str)
common_sizes = stats["image_size"].value_counts().head(6)
stage_mix = (pd.crosstab(stats["image_size"], stats["diagnosis"], normalize="index")
             .reindex(index=common_sizes.index, columns=range(5), fill_value=0) * 100)
print(f"\nstage mix in % of each image size "
      f"(the six most common sizes = {int(common_sizes.sum())} of {len(stats)} images):")
print(stage_mix.round(1).to_string())

# --- Figure 11 --------------------------------------------------------------
fig, axes = plt.subplots(1, 2, figsize=(13, 4.6), gridspec_kw={"width_ratios": [1, 1.4]})

pct = quality_by_stage["low_quality_percent"].reindex(range(5), fill_value=0).to_numpy()
bars = axes[0].bar(range(5), pct, width=0.66, color=STAGE_COLORS)
for bar, value in zip(bars, pct):
    axes[0].text(bar.get_x() + bar.get_width() / 2, value + pct.max() * 0.02, f"{value:.1f}%",
                 ha="center", va="bottom", fontsize=9, color=INK_SOFT)
overall = 100 * stats["low_quality"].mean()
axes[0].axhline(overall, color=INK_SOFT, linewidth=1, linestyle="--")
axes[0].text(0.02, 0.97, f"dashed line: all images ({overall:.1f}%)", transform=axes[0].transAxes,
             ha="left", va="top", fontsize=8.5, color=INK_SOFT)
axes[0].set_xticks(range(5))
axes[0].set_xticklabels([f"{i}\n{CLASS_NAMES[i]}" for i in range(5)])
axes[0].set_ylabel("Dark or blurred (% of that stage)")
axes[0].set_ylim(0, pct.max() * 1.25)
worst_stage = int(np.argmax(pct))
axes[0].set_title(f"Stage {worst_stage} is most often dark or blurred ({pct[worst_stage]:.0f}%)")
tidy(axes[0])

y = np.arange(len(stage_mix))
left = np.zeros(len(stage_mix))
for s in range(5):
    widths = stage_mix[s].to_numpy()
    axes[1].barh(y, widths, left=left, height=0.62, color=STAGE_COLORS[s],
                 edgecolor="#fcfcfb", linewidth=0.6, label=f"{s} {CLASS_NAMES[s]}")
    if s == 0:                                    # label the healthy share, the one that moves most
        for yi, w in zip(y, widths):
            if w >= 8:
                axes[1].text(w / 2, yi, f"{w:.0f}%", ha="center", va="center", fontsize=8.5, color=INK)
    left += widths
axes[1].set_yticks(y)
axes[1].set_yticklabels([f"{size}  (n={n})" for size, n in common_sizes.items()])
axes[1].invert_yaxis()
axes[1].set_xlim(0, 100)
axes[1].set_xlabel("Percent of the images of that size")
axes[1].set_title(f"Stage 0 share by image size: {stage_mix[0].min():.0f}% to {stage_mix[0].max():.0f}%")
axes[1].legend(frameon=False, ncols=5, fontsize=8.5, labelcolor=INK_SOFT,
               loc="upper center", bbox_to_anchor=(0.5, -0.16))
for side in ("top", "right"):
    axes[1].spines[side].set_visible(False)

plt.tight_layout()
plt.savefig(FIG_DIR / "fig11_shortcut_check.png")
plt.show()

## 4. Duplicates

APTOS contains repeated images, and that matters for two reasons. If a photo appears in training
**and** in the test set, the model scores well by recognising a picture it has already seen — that is
data leakage. And a photo that appears several times counts several times towards what the model
learns.

**A first attempt that failed, and why.** The obvious tool is a *difference hash* (dHash): shrink the
image to 9×8 grey pixels, compare each pixel with its right-hand neighbour, and call the resulting 64
bits a fingerprint. On ordinary photographs it works. On this dataset it collapsed: in the first
version of this notebook, a tolerance of 5 differing bits flagged **186,733 pairs** and left only 818
of the 3,662 images. The cause is structural — every fundus photo is a bright disc on black with a
bright optic disc, so at 9×8 pixels they all look like the same picture. The fingerprint described the
shape of the dataset, not the individual eye. The same test is repeated below on this version's
images, because it is a result worth reporting.

**What is used instead — three steps, from certain to careful:**

1. **Identical files.** The md5 of the file's bytes: same bytes, same image, no judgement needed.
   One copy of each is kept and the extras are removed, so a repeated photo cannot dominate training.
   If copies of the same file carry *different* stages, the labelling contradicts itself and all of
   them are dropped.
2. **Near-identical content.** The 128×128 thumbnails from section 3, each already stripped of the
   shape all fundus photos share, compared by correlation. 1.0 means identical content.
3. **Grouping.** Images linked directly, or through a third image, are collected into groups with a
   small union-find structure, so a whole family of near-copies moves together into one split.

**Choosing the threshold — two conditions, both measured on this data.**

*Condition 1: no chains.* Grouping is transitive, which is known as *single linkage*: if A matches B
and B matches C, then A, C and everything else in the chain land in one group even when A and C look
nothing alike. On a synthetic test set a cut of 0.95 was safe, but on the real photographs it chained
64 images into one group with an average internal similarity of only 0.938, while the genuine copies
scored 0.990 and above. So the notebook sweeps the cut from 0.95 to 0.995 and records where the
chains stop. The table is saved as `threshold_sweep.csv`.

*Condition 2: the cut must sit in empty space.* Stopping the chains is not enough. Version 4 simply
took the lowest chain-free cut, 0.97, and its own check (figure 7) showed two borderline pairs being
accepted, at 0.970 and 0.973. In one of them the optic disc is on opposite sides of the two images, so
they cannot be two copies of one photo. Figure 6 shows why it happened: the genuine copies form a
separate block at the top of the ranked curve, and no image at all has its best match between about
0.973 and 0.990. A cut at the edge of the crowd, like 0.97, lets the top of the crowd through. A cut
inside the empty band separates copies from everything else, and a small change in lighting or
compression cannot push an image across it. The final cut is therefore placed **in the middle of the
widest empty band above 0.90**, and the notebook stops if that band is narrower than 0.005, because
that would mean there is no clean separation to find.

The correlation between **every pair** is computed in one matrix multiplication rather than a loop
over millions of pairs.

The notebook also prints the largest groups with the similarity holding them together, shows the
biggest one as a picture, and **stops** if more than 15% of the dataset ends up flagged or any group
exceeds 10 images. A loud failure is better than a quietly ruined dataset.

**How the method got here.** Each row was a real run on Kaggle:

| Version | Method | What happened |
|---|---|---|
| 1 | dHash, up to 5 differing bits | 186,733 pairs flagged; only 818 images left |
| 2 | correlation fingerprint, fixed cut 0.95 | one group of 64 images — the guard stopped the run |
| 3 | identical files removed first, cut 0.95 | still a 64-image chain (internal similarity min 0.908, mean 0.938) |
| 4 | sweep, lowest chain-free cut (0.97) | chains gone, but two pairs of different photos accepted |
| 5 | sweep, then the cut placed in the empty band | this version |

In [ ]:
# --- 1. identical files: keep one copy of each -------------------------------
copies_of_this_file = stats.groupby("md5")["id_code"].transform("size")
stages_for_this_file = stats.groupby("md5")["diagnosis"].transform("nunique")

print("images byte-identical to another image :", int((copies_of_this_file > 1).sum()))
print("distinct photos behind them            :", int(stats.loc[copies_of_this_file > 1, "md5"].nunique()))
print("largest set of identical copies        :", int(copies_of_this_file.max()))

contradictory = stages_for_this_file > 1          # same file, different stage = unusable
first_copy    = ~stats.duplicated(subset="md5", keep="first")
extra_copies  = int((~first_copy & ~contradictory).sum())

print("identical files with disagreeing stages:", int(contradictory.sum()), "(dropped)")
print("extra copies removed                   :", extra_copies)

keep = (first_copy & ~contradictory).to_numpy()
unique_images = stats[keep].reset_index(drop=True)
unique_thumbs = thumbs[keep]
print("images remaining                       :", len(unique_images))

In [ ]:
# --- 2. near-identical content ----------------------------------------------
# Correlation between every remaining pair, in one matrix multiplication.
X = unique_thumbs.reshape(len(unique_thumbs), -1).astype(np.float32)
X -= X.mean(axis=1, keepdims=True)
X /= np.linalg.norm(X, axis=1, keepdims=True) + 1e-8
similarity = X @ X.T
np.fill_diagonal(similarity, -1.0)          # an image is not its own duplicate
best_match = similarity.max(axis=1)         # each image's single closest match

MAX_GROUP = 10          # a duplicate group larger than this means eyes are being chained


def group_images(cut):
    """Group every image whose similarity to another is at least `cut` (union-find)."""
    parent = list(range(len(unique_images)))

    def find(i):
        while parent[i] != i:
            parent[i] = parent[parent[i]]          # path compression keeps this fast
            i = parent[i]
        return i

    pairs = [tuple(map(int, p)) for p in np.argwhere(np.triu(similarity >= cut, k=1))]
    for i, j in pairs:
        root_i, root_j = find(i), find(j)
        if root_i != root_j:
            parent[root_j] = root_i

    ids = np.array([find(i) for i in range(len(unique_images))])
    sizes = pd.Series(ids).value_counts()
    return pairs, ids, sizes


def describe(cut, pairs, sizes):
    """One row of the threshold table."""
    return {"cut": cut, "pairs": len(pairs), "groups": int((sizes > 1).sum()),
            "images_in_groups": int(sizes[sizes > 1].sum()), "largest_group": int(sizes.max()),
            "chains": bool(sizes.max() > MAX_GROUP)}

In [ ]:
# --- condition 1: sweep the cut and see where the chains stop ----------------
rows = []
for cut in (0.95, 0.96, 0.97, 0.98, 0.99, 0.995):
    pairs, _, sizes = group_images(cut)
    rows.append(describe(cut, pairs, sizes))
sweep = pd.DataFrame(rows)
print(sweep.to_string(index=False))

chain_free = sweep.loc[~sweep["chains"], "cut"]
LOWEST_SAFE_CUT = float(chain_free.min()) if len(chain_free) else None
print(f"\nlowest cut without chains: {LOWEST_SAFE_CUT}")

# --- condition 2: put the cut in the empty band -------------------------------
# Only the top of the ranked curve matters: below 0.90 it is sparse and full of gaps
# that mean nothing. 0.90 itself is added as a floor, so that if nothing at all lies
# between 0.90 and the copies, the empty band simply starts at 0.90.
SEARCH_FROM = 0.90
ranked = np.sort(np.append(best_match[best_match >= SEARCH_FROM], SEARCH_FROM))
assert len(ranked) >= 2, "no image has a match above 0.90 - there is nothing to separate"

widest = int(np.argmax(np.diff(ranked)))                 # the biggest jump between neighbours
GAP_LOW, GAP_HIGH = float(ranked[widest]), float(ranked[widest + 1])
SIM_CUT = float(round((GAP_LOW + GAP_HIGH) / 2, 3))

print(f"widest empty band above {SEARCH_FROM}: {GAP_LOW:.3f} to {GAP_HIGH:.3f} "
      f"(width {GAP_HIGH - GAP_LOW:.3f}) - no image has its best match inside it")
print(f"final threshold: {SIM_CUT}, the middle of that band")

# --- the groups at the final cut ----------------------------------------------
pair_list, group_ids, group_sizes = group_images(SIM_CUT)
final_row = describe(SIM_CUT, pair_list, group_sizes)
print(f"at {SIM_CUT}: {final_row['pairs']} pairs, {final_row['groups']} groups, "
      f"largest group {final_row['largest_group']}")

threshold_table = (pd.concat([sweep, pd.DataFrame([final_row])], ignore_index=True)
                   .drop_duplicates(subset="cut", keep="last")
                   .sort_values("cut").reset_index(drop=True))
threshold_table["final"] = threshold_table["cut"] == SIM_CUT
threshold_table.to_csv(OUT_DIR / "threshold_sweep.csv", index=False)

In [ ]:
# What the rejected dHash method says about the same images, for the report
hashes = np.array([np.frombuffer(bytes.fromhex(h), dtype=np.uint8) for h in unique_images["dhash"]])
bits = np.unpackbits(hashes, axis=1).astype(np.int16)
hamming = bits @ (1 - bits).T + (1 - bits) @ bits.T      # differing bits, for every pair at once
np.fill_diagonal(hamming, 999)
dhash_pairs = int(np.triu(hamming <= 5, 1).sum())
print(f"on the same {len(unique_images)} images:")
print(f"   dHash at <= 5 differing bits would flag : {dhash_pairs} pairs")
print(f"   the correlation fingerprint flags        : {len(pair_list)} pairs")

In [ ]:
# Figure 6 - where the threshold sits, and why there
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.4))

# left: every image's best match. Most images have a fairly similar partner even though
# only a handful are copies - which is why the cut has to be strict.
axes[0].hist(best_match, bins=80, color="#2a78d6")
axes[0].axvline(SIM_CUT, color=FLAG, linewidth=2)
axes[0].text(SIM_CUT, axes[0].get_ylim()[1] * 0.92, f" cut {SIM_CUT}", color=FLAG, fontsize=9)
axes[0].set_xlabel("Similarity to the most similar other image")
axes[0].set_ylabel("Images")
axes[0].set_title(f"Even different photos score high (median best match {np.median(best_match):.2f})")
tidy(axes[0])

# right: the top of the ranked curve, one dot per image, with the empty band shaded
top = np.sort(best_match)[::-1][:80]
rank = np.arange(1, len(top) + 1)
axes[1].axhspan(GAP_LOW, GAP_HIGH, color=FLAG, alpha=0.10, linewidth=0)
axes[1].axhline(SIM_CUT, color=FLAG, linewidth=1.6)
axes[1].plot(rank, top, color="#2a78d6", linewidth=1, alpha=0.45)
axes[1].scatter(rank, top, s=16, color="#2a78d6", zorder=3)
axes[1].text(len(top), SIM_CUT, f"cut {SIM_CUT} ", color=FLAG, fontsize=9, ha="right", va="bottom")
axes[1].set_xlabel(f"The {len(top)} images with the strongest best match, ranked")
axes[1].set_ylabel("Similarity to the most similar other image")
axes[1].set_title(f"No image scores between {GAP_LOW:.3f} and {GAP_HIGH:.3f}")
tidy(axes[1])

plt.tight_layout()
plt.savefig(FIG_DIR / "fig06_similarity_threshold.png")
plt.show()

print(f"images whose closest match is >= {SIM_CUT}      : {int((best_match >= SIM_CUT).sum())}")
print(f"images between 0.90 and {SIM_CUT} (not merged) : {int(((best_match >= 0.90) & (best_match < SIM_CUT)).sum())}")

# the empty band must be real, not a rounding accident
assert GAP_HIGH - GAP_LOW >= 0.005, (
    f"the widest empty band is only {GAP_HIGH - GAP_LOW:.4f} wide, so copies and different photos "
    f"are not clearly separated. Look at the right-hand chart above before going further.")

In [ ]:
# Figure 7 and 8 - the pairs either side of the threshold, so the cut can be checked by eye
def show_pairs(pairs, title, filename):
    if not pairs:
        print("nothing to show for:", title)
        return
    fig, axes = plt.subplots(len(pairs), 2, figsize=(7, 3.3 * len(pairs)), squeeze=False)
    for r, (i, j) in enumerate(pairs):
        for c, index in enumerate((i, j)):
            row = unique_images.iloc[index]
            image = cv2.imread(row["path"], cv2.IMREAD_REDUCED_COLOR_8)
            axes[r, c].imshow(cv2.cvtColor(image, cv2.COLOR_BGR2RGB))
            axes[r, c].set_title(f'{row["id_code"]}\nstage {int(row["diagnosis"])} - '
                                 f'{CLASS_NAMES[int(row["diagnosis"])]}', fontsize=9, color=INK_SOFT)
            axes[r, c].axis("off")
        axes[r, 0].text(-0.07, 0.5, f"similarity {similarity[i, j]:.3f}", rotation=90, va="center",
                        ha="center", transform=axes[r, 0].transAxes, fontsize=9, color=INK)
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout()
    plt.savefig(FIG_DIR / filename)
    plt.show()

accepted = sorted(pair_list, key=lambda p: similarity[p[0], p[1]])[:3]
show_pairs(accepted, f"Weakest pairs accepted as copies (cut = {SIM_CUT})",
           "fig07_duplicates_accepted.png")

below = np.argwhere(np.triu((similarity >= 0.85) & (similarity < SIM_CUT), k=1))
rejected = sorted([tuple(map(int, p)) for p in below], key=lambda p: -similarity[p[0], p[1]])[:3]
show_pairs(rejected, f"Strongest pairs rejected as different photos (cut = {SIM_CUT})",
           "fig08_duplicates_rejected.png")

In [ ]:
# --- 3. the groups at the final cut -----------------------------------------
unique_images["group_id"] = group_ids
images_in_groups = int(group_sizes[group_sizes > 1].sum())

print("groups of 2 or more images :", int((group_sizes > 1).sum()))
print("images inside such a group :", images_in_groups,
      f"({100 * images_in_groups / len(unique_images):.1f}% of the remaining images)")
print("largest group              :", int(group_sizes.max()))

# --- what is holding the biggest groups together? ----------------------------
print("\nlargest groups:")
for gid, size in group_sizes.sort_values(ascending=False).head(5).items():
    if size < 2:
        continue
    members = np.where(unique_images["group_id"].to_numpy() == gid)[0]
    sims = [similarity[i, j] for n, i in enumerate(members) for j in members[n + 1:]]
    stages = sorted(unique_images.loc[members, "diagnosis"].unique().tolist())
    print(f"   {size:>3} images | stages {stages} | internal similarity "
          f"min {min(sims):.3f}, mean {np.mean(sims):.3f}")

In [ ]:
# Figure 9 - the biggest group, so a chain of unrelated eyes would be obvious
biggest = group_sizes.idxmax()
all_members = np.where(unique_images["group_id"].to_numpy() == biggest)[0]
members = all_members[:8]
if len(members) > 1:
    stage_list = sorted(unique_images.loc[all_members, "diagnosis"].astype(int).unique().tolist())
    if len(stage_list) > 1:
        title = (f"Largest duplicate group: {len(all_members)} copies of one photo\n"
                 f"labelled with {len(stage_list)} different stages ({', '.join(map(str, stage_list))})")
    else:
        title = f"Largest duplicate group: {len(all_members)} copies of one photo, all stage {stage_list[0]}"

    fig, axes = plt.subplots(1, len(members), figsize=(max(2.3 * len(members), 7), 3.2))
    for ax, index in zip(np.atleast_1d(axes), members):
        row = unique_images.iloc[int(index)]
        ax.imshow(cv2.cvtColor(cv2.imread(row["path"], cv2.IMREAD_REDUCED_COLOR_8), cv2.COLOR_BGR2RGB))
        ax.set_title(f'{row["id_code"][:8]}\nstage {int(row["diagnosis"])}', fontsize=8, color=INK_SOFT)
        ax.axis("off")
    fig.suptitle(title, fontsize=12, fontweight="bold", color=INK)
    plt.tight_layout()
    plt.savefig(FIG_DIR / "fig09_largest_group.png")
    plt.show()
else:
    print("no group larger than one image")

# --- guards: a detector that fires too easily must fail loudly ---------------
assert images_in_groups / len(unique_images) < 0.15, (
    f"{images_in_groups} images ({100*images_in_groups/len(unique_images):.1f}%) were flagged as "
    f"near-duplicates - far more than copies can explain. Check figures 6 to 9 before going on.")
assert group_sizes.max() <= MAX_GROUP, (
    f"one group holds {int(group_sizes.max())} images, so the final cut still chains different "
    f"photos together. Check the figure above and the threshold table before going on.")

# --- 4. do the near-copies agree on the stage? ------------------------------
stages_per_group = unique_images.groupby("group_id")["diagnosis"].nunique()
conflict_groups  = stages_per_group[stages_per_group > 1].index
conflict_mask    = unique_images["group_id"].isin(conflict_groups)
print("\ngroups whose near-copies disagree about the stage:",
      len(conflict_groups), f"({int(conflict_mask.sum())} images, dropped below)")

## 5. The split

The remaining images are divided once: **70% train, 15% validation, 15% test**.

- **Stratified** — each split keeps the same stage proportions, so the rare stages appear everywhere.
- **Group-aware** — the split is made over duplicate *groups*, not images, so near-copies stay together.
- **Fixed** — one seed, saved to `splits.csv`, reused by every later notebook, so experiments are
  comparable and the test set stays untouched until the very end.
- Groups whose copies disagree about the stage are **dropped**; the count goes in the report.

The `assert` lines are deliberate: if a later edit breaks the grouping, the notebook stops here
instead of quietly producing a leaky split.

In [ ]:
from sklearn.model_selection import train_test_split

clean = unique_images[~conflict_mask].copy()
print(f"images going into the split: {len(clean)}")

# one row per GROUP - this is what actually gets split
groups = (clean.groupby("group_id")
          .agg(diagnosis=("diagnosis", "first"), n_images=("id_code", "size"))
          .reset_index())
print("groups to split:", len(groups))

train_groups, rest = train_test_split(
    groups, test_size=0.30, stratify=groups["diagnosis"], random_state=SEED)
val_groups, test_groups = train_test_split(
    rest, test_size=0.50, stratify=rest["diagnosis"], random_state=SEED)

split_of = {}
for frame, name in ((train_groups, "train"), (val_groups, "val"), (test_groups, "test")):
    split_of.update({gid: name for gid in frame["group_id"]})
clean["split"] = clean["group_id"].map(split_of)

# --- checks that must pass ---------------------------------------------------
assert clean["split"].notna().all(), "some images were not assigned to a split"
assert clean.groupby("group_id")["split"].nunique().max() == 1, \
    "a duplicate group ended up in more than one split"
assert set(clean["split"]) == {"train", "val", "test"}

print("\nimages per split:")
print(clean["split"].value_counts().reindex(["train", "val", "test"]).to_string())

In [ ]:
# Stage proportions per split - the proof that stratification worked
pivot = (clean.pivot_table(index="diagnosis", columns="split",
                           values="id_code", aggfunc="count")
         .reindex(columns=["train", "val", "test"]).fillna(0).astype(int))
percent = (100 * pivot / pivot.sum()).round(1)

print("images per stage and split:")
print(pivot.to_string())
print("\npercent within each split:")
print(percent.to_string())

x, width = np.arange(5), 0.26
fig, ax = plt.subplots(figsize=(7.6, 4.4))
for k, split in enumerate(["train", "val", "test"]):
    bars = ax.bar(x + (k - 1) * width, percent[split], width=width - 0.03,
                  color=SPLIT_COLORS[split], label=split)
    for bar, value in zip(bars, percent[split]):
        ax.text(bar.get_x() + bar.get_width()/2, value + 0.7, f"{value:.0f}",
                ha="center", fontsize=8, color=INK_SOFT)

ax.set_xticks(x)
ax.set_xticklabels([f"{i}\n{CLASS_NAMES[i]}" for i in range(5)])
ax.set_ylabel("Percent of that split")
ax.set_ylim(0, float(percent.to_numpy().max()) * 1.20)
ax.set_title("Stage proportions match across train, validation and test")
ax.legend(frameon=False, labelcolor=INK_SOFT, ncols=3)
tidy(ax)
plt.savefig(FIG_DIR / "fig10_split_distribution.png")
plt.show()

## 6. Save the results

Four files go to `/kaggle/working`:

- **`splits.csv`** — one row per image: id, stage, split, group, the measurements and the low-quality flag. This is the file every later notebook reads.
- **`image_stats_all.csv`** — the full measurements including the dropped images, kept for the report.
- **`threshold_sweep.csv`** — the threshold table from section 4 (saved when it was made), for a table in the report.
- **`dataset_summary.json`** — the headline numbers, so the report quotes the run rather than memory.

Plus the figures in `/kaggle/working/figures/`.

In [ ]:
columns = ["id_code", "diagnosis", "split", "group_id", "width", "height", "crop_fraction",
           "brightness", "contrast", "sharpness", "low_quality", "md5"]
clean[columns].sort_values("id_code").to_csv(OUT_DIR / "splits.csv", index=False)
stats.to_csv(OUT_DIR / "image_stats_all.csv", index=False)

summary = {
    "images_total": int(len(stats)),
    "images_kept": int(len(clean)),
    "exact_copies_removed": extra_copies,
    "identical_files_with_disagreeing_stages": int(contradictory.sum()),
    "class_counts_total": {CLASS_NAMES[int(k)]: int(v) for k, v in counts.items()},
    "byte_identical_images": int((copies_of_this_file > 1).sum()),
    "near_duplicate_pairs": int(len(pair_list)),
    "images_in_duplicate_groups": images_in_groups,
    "near_duplicate_groups": int((group_sizes > 1).sum()),
    "largest_group": int(group_sizes.max()),
    "conflicting_groups_dropped": int(len(conflict_groups)),
    "images_dropped": int(len(stats) - len(clean)),
    "low_quality_flagged": int(low_quality.sum()),
    "split_sizes": {k: int(v) for k, v in clean["split"].value_counts().items()},
    "similarity_threshold_chosen": SIM_CUT,
    "lowest_cut_without_chains": LOWEST_SAFE_CUT,
    "empty_band": [round(GAP_LOW, 4), round(GAP_HIGH, 4)],
    "max_group_allowed": MAX_GROUP,
    "dhash_pairs_if_used": dhash_pairs,
    "seed": SEED,
}
with open(OUT_DIR / "dataset_summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps(summary, indent=2))
print("\nfiles written:", sorted(p.name for p in OUT_DIR.glob("*.csv")),
      "+ dataset_summary.json +", len(list(FIG_DIR.glob("*.png"))), "figures")

## 7. What to do next

1. **Save Version → Save & Run All.** This runs the notebook from top to bottom on a clean machine and stores the outputs. Only a saved version's output can be attached to the next notebook.
2. Send back `dataset_summary.json` and `threshold_sweep.csv`. Those numbers go straight into the report's dataset section.
3. Download the notebook (**File → Download notebook**) into `CW1_DR_Stage_Detection/kaggle_notebooks/` and commit it.

**For the report**, this notebook produces: the class distribution figure and table, the image size figure, the sample grid, the quality figure and the worst-quality examples, the shortcut check, the threshold table and figure, the duplicate examples, and the split proportions figure — with the summary numbers to describe them.

Notebook 2 will take `splits.csv` from here and build the preprocessing pipeline.